# ELECTRA: Efficiently Learning an Encoder that Classifies Token Replacements Accurately

ELECTRA is a novel pre-training method for language models that focuses on replaced token detection rather than masked language modeling (MLM).

## Key Components
1. Generator (G): A small masked language model
2. Discriminator (D): A binary classification model

## Mathematical Formulation

### 1. Generator Objective
$$\mathcal{L}_G = - \sum_{i \in \mathcal{M}} \log P_G(x_i | \tilde{x})$$
- $\mathcal{M}$: set of masked token positions
- $P_G(x_i | \tilde{x})$: probability of the original token given the masked sequence

### 2. Discriminator Objective
$$\mathcal{L}_D = - \sum_{i=1}^n \left[ \mathbb{I}(x_i = \tilde{x}'_i) \log P_D(\tilde{x}'_i = x_i | \tilde{x}') + \mathbb{I}(x_i \neq \tilde{x}'_i) \log (1 - P_D(\tilde{x}'_i = x_i | \tilde{x}')) \right]$$
- $\mathbb{I}$: indicator function
- $P_D(\tilde{x}'_i = x_i | \tilde{x}')$: probability that $\tilde{x}'_i$ is the original token

### 3. Joint Training Objective
$$\mathcal{L} = \mathcal{L}_D + \lambda \mathcal{L}_G$$
- $\lambda$: hyperparameter balancing generator and discriminator losses

## Training Process
1. Pre-training:
   - Generator pre-trained using MLM
   - Joint training of both networks
2. Fine-tuning:
   - Generator discarded
   - Discriminator fine-tuned on downstream tasks

## Efficiency and Performance
- Utilizes all tokens in the sequence, not just masked ones
- Faster training and improved resource utilization
- Achieves state-of-the-art performance on various NLP benchmarks


### Citations and References

- Clark, K., Luong, M. T., Le, Q. V., & Manning, C. D. (2020). ELECTRA: Pre-training Text Encoders as Discriminators Rather Than Generators. In *International Conference on Learning Representations (ICLR)*. Retrieved from https://openreview.net/forum?id=r1xMH1BtvB

- Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. In *Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies, Volume 1 (Long and Short Papers)* (pp. 4171-4186). doi:10.18653/v1/N19-1423

- Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language Models are Unsupervised Multitask Learners. Retrieved from https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

These references provide a comprehensive background on ELECTRA and its place within the broader context of pre-trained language models.

In [1]:
!pip install torch==2.3.1 transformers

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import BertTokenizer, BertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the Generator model
class Generator(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(Generator, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        prediction_scores = self.linear(sequence_output)
        return prediction_scores

# Define the Discriminator model
class Discriminator(nn.Module):
    def __init__(self, hidden_size):
        super(Discriminator, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        logits = self.linear(sequence_output).squeeze(-1)
        return logits

# Prepare dataset (dummy dataset for illustration)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
sentences = ["This is an example sentence.", "Here is another one."]
inputs = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True)
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

# Masking Function
def mask_tokens(input_ids, tokenizer, mask_prob=0.15):
    labels = input_ids.clone()
    probability_matrix = torch.full(labels.shape, mask_prob)
    special_tokens_mask = [
        tokenizer.get_special_tokens_mask(val, already_has_special_tokens=True) for val in labels.tolist()
    ]
    probability_matrix.masked_fill_(torch.tensor(special_tokens_mask, dtype=torch.bool), value=0.0)
    masked_indices = torch.bernoulli(probability_matrix).bool()
    labels[~masked_indices] = -100  # We only compute loss on masked tokens

    indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
    input_ids[indices_replaced] = tokenizer.convert_tokens_to_ids(tokenizer.mask_token)

    indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
    random_words = torch.randint(len(tokenizer), labels.shape, dtype=torch.long)
    input_ids[indices_random] = random_words[indices_random]

    return input_ids, labels

# Apply masking to input data
input_ids, labels = mask_tokens(input_ids, tokenizer)

# Hyperparameters
vocab_size = tokenizer.vocab_size
hidden_size = 768
lambda_gen = 50  # Balance between generator and discriminator losses
lr = 1e-4
num_epochs = 60
clip_value = 1.0  # Gradient clipping value

# Initialize models, loss functions, and optimizers
generator = Generator(vocab_size, hidden_size)
discriminator = Discriminator(hidden_size)
criterion_gen = nn.CrossEntropyLoss()
criterion_disc = nn.BCEWithLogitsLoss()
optimizer_gen = optim.Adam(generator.parameters(), lr=lr, weight_decay=1e-5)
optimizer_disc = optim.Adam(discriminator.parameters(), lr=lr, weight_decay=1e-5)

# Training loop
for epoch in range(num_epochs):
    generator.train()
    discriminator.train()

    # Generator forward pass
    outputs_gen = generator(input_ids, attention_mask=attention_mask)
    mask_token_index = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)

    if mask_token_index[0].size(0) == 0:
        print("No masked tokens found, skipping epoch.")
        continue

    logits_gen = outputs_gen[mask_token_index]
    labels_gen = labels[mask_token_index]

    loss_gen = criterion_gen(logits_gen, labels_gen)
    optimizer_gen.zero_grad()
    loss_gen.backward()
    torch.nn.utils.clip_grad_norm_(generator.parameters(), clip_value)
    optimizer_gen.step()

    # Replace masked tokens with generator's predictions
    _, predicted_tokens = torch.max(logits_gen, dim=-1)
    input_ids_replaced = input_ids.clone()
    input_ids_replaced[mask_token_index] = predicted_tokens

    # Discriminator forward pass
    logits_disc = discriminator(input_ids_replaced, attention_mask=attention_mask)
    labels_disc = (input_ids != input_ids_replaced).float()
    loss_disc = criterion_disc(logits_disc, labels_disc)

    optimizer_disc.zero_grad()
    loss_disc.backward()
    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), clip_value)
    optimizer_disc.step()

    total_loss = loss_disc + lambda_gen * loss_gen

    print(f"Epoch {epoch+1}/{num_epochs}, Loss Gen: {loss_gen.item()}, Loss Disc: {loss_disc.item()}, Total Loss: {total_loss.item()}")

print("Training complete.")

# Evaluation
def evaluate_model(generator, discriminator, input_ids, attention_mask):
    generator.eval()
    discriminator.eval()

    with torch.no_grad():
        outputs_gen = generator(input_ids, attention_mask=attention_mask)
        mask_token_index = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)

        if mask_token_index[0].size(0) == 0:
            print("No masked tokens found in evaluation data.")
            return

        logits_gen = outputs_gen[mask_token_index]
        _, predicted_tokens = torch.max(logits_gen, dim=-1)
        input_ids_replaced = input_ids.clone()
        input_ids_replaced[mask_token_index] = predicted_tokens

        logits_disc = discriminator(input_ids_replaced, attention_mask=attention_mask)
        predictions = torch.sigmoid(logits_disc).round()
        labels_disc = (input_ids != input_ids_replaced).float()

        y_true = labels_disc.view(-1).cpu().numpy()
        y_pred = predictions.view(-1).cpu().numpy()

        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        print(f"Accuracy: {accuracy}")
        print(f"Precision: {precision}")
        print(f"Recall: {recall}")
        print(f"F1 Score: {f1}")

# Prepare evaluation data (dummy data for illustration)
eval_sentences = ["This is a test sentence.", "Another evaluation sentence."]
eval_inputs = tokenizer(eval_sentences, return_tensors='pt', padding=True, truncation=True)
eval_input_ids = eval_inputs['input_ids']
eval_attention_mask = eval_inputs['attention_mask']
eval_input_ids, eval_labels = mask_tokens(eval_input_ids, tokenizer)

# Evaluate the model
evaluate_model(generator, discriminator, eval_input_ids, eval_attention_mask)


Epoch 1/60, Loss Gen: 10.434524536132812, Loss Disc: 0.7590258121490479, Total Loss: 522.4852294921875
Epoch 2/60, Loss Gen: 8.925592422485352, Loss Disc: 0.3740732967853546, Total Loss: 446.6537170410156
Epoch 3/60, Loss Gen: 7.013761520385742, Loss Disc: 0.19232739508152008, Total Loss: 350.8804016113281
Epoch 4/60, Loss Gen: 5.481874942779541, Loss Disc: 0.12148217856884003, Total Loss: 274.2152404785156
Epoch 5/60, Loss Gen: 4.316492557525635, Loss Disc: 0.0748487338423729, Total Loss: 215.89947509765625
Epoch 6/60, Loss Gen: 3.598348379135132, Loss Disc: 0.05262249708175659, Total Loss: 179.9700469970703
Epoch 7/60, Loss Gen: 3.020427703857422, Loss Disc: 0.03109876438975334, Total Loss: 151.052490234375
Epoch 8/60, Loss Gen: 2.629405975341797, Loss Disc: 0.01420238520950079, Total Loss: 131.48451232910156
Epoch 9/60, Loss Gen: 2.0910961627960205, Loss Disc: 0.007507426664233208, Total Loss: 104.56231689453125
Epoch 10/60, Loss Gen: 1.9106149673461914, Loss Disc: 0.004381971433758

## Comparison of Provided Script and ELECTRA Model

### Overview of ELECTRA

ELECTRA (Efficiently Learning an Encoder that Classifies Token Replacements Accurately) enhances traditional masked language modeling by employing a generator-discriminator framework. Here's an overview of its innovations:

1. **Generator and Discriminator**:
   - **Generator**: Replaces masked tokens with plausible alternatives.
   - **Discriminator**: Differentiates between real tokens and replacements generated by the generator.

2. **Replaced Token Detection**:
   - ELECTRA masks a portion of tokens and generates replacements. The discriminator is trained to classify tokens as either real or replaced.

3. **Training Efficiency**:
   - ELECTRA is designed for improved efficiency compared to models that predict each masked token individually.

### Differences Between Script and ELECTRA

#### Model Architecture

- **Generator**:
  - **Script**: Utilizes a BERT-based model with a linear layer for token prediction.
  - **ELECTRA**: Similar architecture but with enhanced token generation techniques.

- **Discriminator**:
  - **Script**: Applies a BERT-based model with a linear layer to classify tokens as real or replaced.
  - **ELECTRA**: Employs a BERT-based model with advanced training methods.

#### Loss Calculation

- **Generator Loss**:
  - **Script**: Computes cross-entropy loss for token prediction using: \\
   
$\text{Loss}_{\text{gen}}$ = $ -\sum_{i} \log \left( \frac{e^{\text{logits}_{i}}}{\sum_{j} e^{\text{logits}_{j}}} \right) $


where $\text{logits}_{i}$ represents the prediction for the $i$-th token.
  
  - **ELECTRA**: Integrates losses for the generator and discriminator: \\
    \text{Loss}_{\text{total}} = \text{Loss}_{\text{gen}} + \lambda \cdot \text{Loss}_{\text{disc}} \\
    where $\text{Loss}_{\text{disc}}$ is the discriminator loss, and $\lambda$ balances the two losses.

- **Discriminator Loss**:
  - **Script**: Uses binary cross-entropy loss:
    \text{Loss}_{\text{disc}} = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(p_i) + (1 - y_i) \log(1 - p_i)] \\
    
    where $y_i$ is the true label, and $p_i$ is the predicted probability for the $i$-th token.

  - **ELECTRA**: Uses more complex loss functions integrating both generator and discriminator outputs.

#### Replacement Strategy

- **Script**:
  - Replaces masked tokens with `[MASK]` or random tokens. This is a simplified approach.

- **ELECTRA**:
  - Utilizes a refined replacement strategy where the generator produces more plausible replacements for tokens, enhancing discriminator training.

#### Training Efficiency

- **Script**:
  - The script may not fully leverage ELECTRA's efficiency due to simplified token replacement and loss calculation methods.

- **ELECTRA**:
  - Designed to be more efficient by reducing the need for extensive training data and improving training dynamics with its generator-discriminator setup.

### Summary

While the provided script captures the basic structure of ELECTRA, it lacks some of the model's advanced features. Improvements to better align with ELECTRA would include:

- **Advanced Token Replacement**: Implementing a more sophisticated replacement strategy for the generator.
- **Integrated Loss Functions**: Using complex loss functions that consider both generator and discriminator outputs.
- **Enhanced Training Efficiency**: Adopting methods to improve computational efficiency and training dynamics.

The script offers a simplified version and does not fully reflect the nuances of ELECTRA's training and evaluation processes.